## 8.1 语法错误

In [ ]:
# 语法错误又称为解析错误
while True print('Hello world')


SyntaxError: invalid syntax (3511958662.py, line 1)

## 8.2 异常

In [2]:
# zerodivisionerror
10 * (1/0)

ZeroDivisionError: division by zero

In [3]:
# NameError
x=10
print(x)
print(y)


10


NameError: name 'y' is not defined

In [5]:
# TypeError
 10 + '5'
 

IndentationError: unexpected indent (791655267.py, line 2)

zero division error\name error\type error都是常见的内置异常，通常列出了内置异常及其含义


## 8.3 异常的处理 
try 语句的使用

In [7]:
while True:
    try:
        x = int(input("Please enter a number: "))
        break
    except ValueError:
        print("Oops!  That was no valid number.  Try again...")

Oops!  That was no valid number.  Try again...


try语句的工作原理：

- 首先，执行 try 子句 （try 和 except 关键字之间的（多行）语句）。

- 如果没有触发异常，则跳过 except 子句，try 语句执行完毕。

- 如果在执行 try 子句时发生了异常，则跳过该子句中剩下的部分。 如果异常的类型与 except 关键字后指定的异常相匹配，则会执行 except 子句，然后跳到 try/except 代码块之后继续执行。

- 如果发生的异常与 except 子句 中指定的异常不匹配，则它会被传递到外层的 try 语句中；如果没有找到处理器，则它是一个 未处理异常 且执行将停止并输出一条错误消息。

In [10]:
# try 语句可以有多个except 子句来为不同的异常指定处理程序
# 一个 except 子句中的类匹配的异常将是该类本身的实例或其所派生的类的实例（但反过来则不可以 --- 列出派生类的 except 子句 不会匹配其基类的实例）
class B(Exception):
    pass

class C(B):
    pass

class D(C):
    pass

for cls in [B, C, D]:
    try:
        raise cls()
    except D:
        print("D")
    except C:
        print("C")
    except B:
        print("B")

# 如果反过来，只能输出BBB
for cls in [B, C, D]:
    try:
        raise cls()
    except B:
        print("B")
    except C:
        print("C")
    except D:
        print("D")
    

B
C
D
B
B
B


In [11]:
# except 子句 可能会在异常名称后面指定一个变量
try:
    raise Exception('spam', 'eggs')
except Exception as inst: # inst 是异常实例
    print(type(inst))    # 异常的类型
    print(inst.args)     # 参数保存在 .args 中
    print(inst)          # __str__ 允许 args 被直接打印，
                         # 但可能在异常子类中被覆盖
    x, y = inst.args     # 解包 args
    print('x =', x)
    print('y =', y)

<class 'Exception'>
('spam', 'eggs')
('spam', 'eggs')
x = spam
y = eggs


exception类的继承关系： BaseException 是所有异常的共同基类。它的一个子类， Exception ，是所有非致命异常的基类。不是 Exception 的子类的异常通常不被处理，因为它们被用来指示程序应该终止。

In [14]:
import sys

try:
    f = open('text.txt')
    s = f.readline()
    i = int(s.strip())
except OSError as err:
    print("OS error:", err)
except ValueError:
    print("Could not convert data to an integer.")
except Exception as err:
    print(f"Unexpected {err=}, {type(err)=}")
    raise

Could not convert data to an integer.


try ... except 语句具有可选的 else 子句，该子句如果存在，它必须放在所有 except 子句 之后。 它适用于 try 子句 没有引发异常但又必须要执行的代码。

In [15]:
for arg in sys.argv[1:]:
    try:
        f = open(arg, 'r')
    except OSError:
        print('cannot open', arg)
    else:
        print(arg, 'has', len(f.readlines()), 'lines')
        f.close()

cannot open --f=c:\Users\ZYF\AppData\Roaming\jupyter\runtime\kernel-v3a305721caa9520fdaf1d00bac3aaa1a1c94ed921.json


异常处理程序不仅会处理在 try 子句 中立刻发生的异常，还会处理在 try 子句 中调用（包括间接调用）的函数。

In [16]:
def this_fails():
    x = 1/0

try:
    this_fails()
except ZeroDivisionError as err:
    print('Handling run-time error:', err)

Handling run-time error: division by zero


## 8.4 触发异常

raise语句

In [17]:
raise NameError('HiThere')

NameError: HiThere

In [18]:
# 如果只想判断是否触发了异常，但并不打算处理该异常，则可以使用更简单的 raise 语句重新触发异常：
try: 
    raise NameError('HiHere')
except NameError:
    print('An exception flew by ')
    raise

An exception flew by 


NameError: HiHere

## 8.5 异常链

In [ ]:
# 隐式链的根因
try:
    open("database.sqlite")
except OSError:
    raise RuntimeError("unable to handle error")

RuntimeError: unable to handle error

In [21]:
# 显式链的根因
def func():
    raise ConnectionError

try:
    func()
except ConnectionError as exc:
    raise RuntimeError('Failed to open database') from exc

RuntimeError: Failed to open database

In [24]:
import requests
import akshare as ak

# 自定义金融业务异常（统一封装股票数据错误）
class StockDataLoadError(Exception):
    """股票数据加载业务异常"""
    def __init__(self, stock_code: str, detail: str):
        self.stock_code = stock_code
        super().__init__(f"股票[{stock_code}]加载失败：{detail}")

# 加载股票日线的函数
def load_stock_daily(stock_code: str, start_date: str, end_date: str):
    try:
        # 模拟akshare请求超时（真实场景下就是网络慢）
        df = ak.stock_zh_a_hist(
            symbol=stock_code, period="daily",
            start_date=start_date, end_date=end_date,
            timeout=0.1  # 故意设短超时触发异常
        )
        return df
    except requests.exceptions.Timeout as e:
        # 【核心：显式链！】把超时异常作为根因，封装成业务异常
        raise StockDataLoadError(stock_code, "日线接口请求超时") from e

# 调用
if __name__ == "__main__":
    load_stock_daily("000001", "20240101", "20240131")

StockDataLoadError: 股票[000001]加载失败：日线接口请求超时

In [25]:
# 禁用自动隐藏链
try:
    open('database.sqlite')
except OSError:
    raise RuntimeError from None

RuntimeError: 

# 8.6 用户自定义异常

程序可以通过创建新的异常类命名自己的异常（Python 类的内容详见 类）。不论是以直接还是间接的方式，异常都应从 Exception 类派生

大多数异常命名都以 “Error” 结尾，类似标准异常的命名。

## 8.7 定义清理操作

try 语句还有一个可选子句finally，用于定义在所有情况下都必须要执行的清理操作。

In [27]:
try:
    raise KeyboardInterrupt from None
finally:
    print('bye')


bye


KeyboardInterrupt: 

几种比较复杂的触发异常情景：

- 如果执行 try 子句期间触发了某个异常，则某个 except 子句应处理该异常。如果该异常没有 except 子句处理，在 finally 子句执行后会被重新触发。

- except 或 else 子句执行期间也会触发异常。 同样，该异常会在 finally 子句执行之后被重新触发。

- 如果 finally 子句中包含 break、continue 或 return 等语句，异常将不会被重新引发。

- 如果执行 try 语句时遇到 break,、continue 或 return 语句，则 finally 子句在执行 break、continue 或 return 语句之前执行。

- 如果 finally 子句中包含 return 语句，则返回值来自 finally 子句的某个 return 语句的返回值，而不是来自 try 子句的 return 语句的返回值。

In [ ]:
def bool_return():
    try:
        return True
    finally:
        return False # finally子句中包含return语句

bool_return()

False

In [29]:
def divide(x,y):
    try :
        result=x/y
    except ZeroDivisionError:
        print('division by zero')
    else:
        print('result is', result)
    finally:
        print('executing finally clause')

In [30]:
divide(1,2)

result is 0.5
executing finally clause


In [31]:
divide(1,0)

division by zero
executing finally clause


In [32]:
divide('2','1') # 未被处理的异常，遇到finally子句会重新抛出

executing finally clause


TypeError: unsupported operand type(s) for /: 'str' and 'str'

## 8.8 预定义的清理操作

某些对象定义了不需要该对象时要执行的标准清理操作。无论使用该对象的操作是否成功，都会执行清理操作。

with 语句支持以及时、正确的清理的方式使用文件对象：

In [33]:
with open("text.txt") as f:
    for line in f:
        print(line, end="")

这是明朝的臣子


## 8.9 引发和处理多个不想关的异常--exceptionGroup

In [34]:
def f():
    excs = [OSError('error 1'), SystemError('error 2')]
    raise ExceptionGroup('there were problems', excs)

f()

  + Exception Group Traceback (most recent call last):
  |   File "C:\Users\ZYF\AppData\Roaming\Python\Python313\site-packages\IPython\core\interactiveshell.py", line 3699, in run_code
  |     exec(code_obj, self.user_global_ns, self.user_ns)
  |     ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  |   File "C:\Users\ZYF\AppData\Local\Temp\ipykernel_40496\1521594820.py", line 5, in <module>
  |     f()
  |     ~^^
  |   File "C:\Users\ZYF\AppData\Local\Temp\ipykernel_40496\1521594820.py", line 3, in f
  |     raise ExceptionGroup('there were problems', excs)
  | ExceptionGroup: there were problems (2 sub-exceptions)
  +-+---------------- 1 ----------------
    | OSError: error 1
    +---------------- 2 ----------------
    | SystemError: error 2
    +------------------------------------


In [35]:
try:
    f()
except Exception as e:
    print(f'caught {type(e)} :{e}')

caught <class 'ExceptionGroup'> :there were problems (2 sub-exceptions)


使用except* 代替 except ，我们可以有选择地只处理组中符合某种类型的异常

In [37]:
def f():
    raise ExceptionGroup(
        'group1',
        [
            OSError('error 1'),
            SystemError('error 2'),
            ExceptionGroup(
                'group2',
                [
                    OSError('error 3'),
                    RecursionError('error 4')
                ]
            )
        ]
    )

In [38]:
try:
    f()
except* OSError as e:
    print("There were OSErrors")
except* SystemError as e:
    print("There were SystemErrors")

There were OSErrors
There were SystemErrors


  + Exception Group Traceback (most recent call last):
  |   File "C:\Users\ZYF\AppData\Roaming\Python\Python313\site-packages\IPython\core\interactiveshell.py", line 3699, in run_code
  |     exec(code_obj, self.user_global_ns, self.user_ns)
  |     ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  |   File "C:\Users\ZYF\AppData\Local\Temp\ipykernel_40496\374190754.py", line 2, in <module>
  |     f()
  |     ~^^
  |   File "C:\Users\ZYF\AppData\Local\Temp\ipykernel_40496\1376688780.py", line 2, in f
  |     raise ExceptionGroup(
  |     ...<12 lines>...
  |     )
  | ExceptionGroup: group1 (1 sub-exception)
  +-+---------------- 1 ----------------
    | ExceptionGroup: group2 (1 sub-exception)
    +-+---------------- 1 ----------------
      | RecursionError: error 4
      +------------------------------------


In [42]:
excs = []
for test in tests:
    try:
        test.run()
    except Exception as e:
        excs.append(e) # 收集所有异常

if excs:
    raise ExceptionGroup('Test failures', excs)# 如果有异常，抛出异常组


NameError: name 'tests' is not defined

## 8.10 用注释细化异常情况

 add_note(note) 方法为异常添加注释

In [44]:
try:
    raise TypeError('bad type')
except Exception as e:
    e.add_note('add some information')
    e.add_note('add some more information')
    raise

TypeError: bad type

In [45]:
def f():
    raise OSError('operation failed')

excs=[]
for i in range(3):
    try:
        f()
    except Exception as e:
        e.add_note(f'attempt {i+1} failed')
        excs.append(e)
    
raise ExceptionGroup('Operation failures', excs)

  + Exception Group Traceback (most recent call last):
  |   File "C:\Users\ZYF\AppData\Roaming\Python\Python313\site-packages\IPython\core\interactiveshell.py", line 3699, in run_code
  |     exec(code_obj, self.user_global_ns, self.user_ns)
  |     ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  |   File "C:\Users\ZYF\AppData\Local\Temp\ipykernel_40496\2346617220.py", line 12, in <module>
  |     raise ExceptionGroup('Operation failures', excs)
  | ExceptionGroup: Operation failures (3 sub-exceptions)
  +-+---------------- 1 ----------------
    | Traceback (most recent call last):
    |   File "C:\Users\ZYF\AppData\Local\Temp\ipykernel_40496\2346617220.py", line 7, in <module>
    |     f()
    |     ~^^
    |   File "C:\Users\ZYF\AppData\Local\Temp\ipykernel_40496\2346617220.py", line 2, in f
    |     raise OSError('operation failed')
    | OSError: operation failed
    | attempt 1 failed
    +---------------- 2 ----------------
    | Traceback (most recent call last):
    |  